In [ ]:
%load_ext autoreload
%autoreload 2

# Build dataloader

In [1]:
import yaml
from pprint import pprint

from torch.utils.data import DataLoader

from probing.finebio import FineBioDataset
from probing.train_utils import decode_clip, fix_random_seed

# load config
with open("configs/probing_atomic_operation.yaml", 'r') as f:
    cfg = yaml.safe_load(f)
dataset_cfg = cfg.get("dataset")
cls_cfg = cfg.get('classifier')
opt_cfg = cfg.get('opt')
assert opt_cfg is not None, 'Missing opt config.'
assert cls_cfg is not None, 'Missing classifier config.'
assert dataset_cfg is not None, "Unvalid config! Missing 'dataset' key."
pprint(cfg)
# build train dataset
train_dataset = FineBioDataset(split=["train"], **dataset_cfg)
rng_generator = fix_random_seed(cfg['init_rand_seed'], include_cuda=True)

# build train loader
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=8,
    collate_fn=decode_clip,
    drop_last=True,
    generator=rng_generator
)

val_dataset = FineBioDataset(split=['valid'], **dataset_cfg)
val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=8,
    collate_fn=decode_clip,
    drop_last=False,
)

{'classifier': {'embed_dim': 1024, 'num_blocks': 4, 'num_heads': 16},
 'dataset': {'allow_clip_overlap': True,
             'frames_per_clip': 16,
             'json_file': 'data/annotations/annotation_all.json',
             'label2id_dir': 'data/annotations',
             'random_view': False,
             'sampling_rate': 1,
             'video_dir': ['data/FineBio/03 finebio_videos_fpv_all_w640',
                           'data/FineBio/10 '
                           'finebio_videos_tpv_all_w640/finebio_videos_w640']},
 'init_rand_seed': 2708,
 'opt': {'lr': 0.000425, 'warmup': 5, 'wd': 0.04}}


# Load V-JEPA 2

In [2]:
from probing.models import init_vjepa2

model, processor = init_vjepa2()

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

model total parameters: 325.97M


In [3]:
model.config.hidden_size

1024

# Build Attentive Classifier

In [4]:
# load classifier
from probing.classifiers import init_classifier

NUM_CLASSES = {k: v['num_classes'] for k,v in train_dataset.type_info.items()}
print(NUM_CLASSES)

classifier = init_classifier(
    num_verb_classes=NUM_CLASSES['verb'],
    num_mani_classes=NUM_CLASSES['manipulated'],
    num_affect_classes=NUM_CLASSES['affected'],
    **cls_cfg
)

{'verb': 10, 'manipulated': 35, 'affected': 36, 'atomic_operation': 244, 'hand': 2}


/home/tomtom/miniconda3/envs/k-pro/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [5]:
classifier.embed_dim

1024

# Train probing layer

In [ ]:
import wandb
import time
import os
import torch
import torch.nn as nn

from probing.losses import sigmoid_focal_loss
from probing.optimizers import init_opt
from probing.train_utils import train_one_epoch, valid_one_epoch

epochs = 40
lr = opt_cfg.get('lr')
wd = opt_cfg.get('wd')
warmup = opt_cfg.get('warmup', 5)
save_freq = 5
ckpt_dir = 'ckpt'
if os.path.exists(ckpt_dir):
    os.mkdir(ckpt_dir)
# criterion = nn.CrossEntropyLoss()
# criterion = sigmoid_focal_loss
optimizer, scheduler, wd_scheduler, _ = init_opt(
    params=classifier.parameters(), 
    iterations_per_epoch=len(train_dataset)//16,
    num_epochs=epochs,
    warmup=warmup,
    lr=lr,
    start_lr=lr,
    final_lr=lr,
    weight_decay=wd,
    final_weight_decay=wd,
)
logger = wandb.init(
    project='vjepa2-probing',
    name=f'debug-{int(time.time())}',
    mode='online',
    reinit='create_new'
)
for epoch in range(epochs):
    train_one_epoch(
        model=model,
        processor=processor,
        classifier=classifier,
        optimizer=optimizer,
        scheduler=scheduler,
        wd_scheduler=wd_scheduler,
        criterion=sigmoid_focal_loss,
        data_loader=train_loader,
        num_verb_classes=NUM_CLASSES['verb'],
        num_mani_classes=NUM_CLASSES['manipulated'],
        num_affect_classes=NUM_CLASSES['affected'],
        logger=logger,
    )
    # save model weights every save freq
    if (epoch+1)%save_freq == 0:
        save_path = os.path.join(ckpt_dir, f"{epoch+1}.pt")
        torch.save({
            "epoch": epoch,
            "classifier": classifier.state_dict(),
            # "optimizer": optimizer.state_dict(),
        }, save_path)
    
    valid_one_epoch(
        model=model,
        processor=processor,
        classifier=classifier,
        criterion=sigmoid_focal_loss,
        data_loader=val_loader,
        num_verb_classes=NUM_CLASSES['verb'],
        num_mani_classes=NUM_CLASSES['manipulated'],
        num_affect_classes=NUM_CLASSES['affected'],
        logger=logger,
    )
logger.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/tomtom/.netrc.
wandb: Currently logged in as: tangptnhan (tangptnhan-kyushu-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


  0%|          | 0/4973 [00:00<?, ?it/s]/home/tomtom/miniconda3/envs/k-pro/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/home/tomtom/miniconda3/envs/k-pro/lib/python3.12/site-packages/torch/autograd/graph.py:865: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:897.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
  8%|▊         | 418/4973 [14:03<2:33:30,  2.02s/it, loss=7.2659, verb_loss=3.0025, manip_loss=1.6132, aff_loss=2.6501]    

# Visualize segment

In [ ]:
take_list = []
open_list = []
press_list = []
release_list = []
insert_list = []

for i in range(len(trainset)):
    
    if id2verb[int(trainset[i]["verb"])] == "take":
        take_list.append(trainset[i])
    if id2verb[int(trainset[i]["verb"])] == "open":
        open_list.append(trainset[i])
    if id2verb[int(trainset[i]["verb"])] == "press":
        press_list.append(trainset[i])
    if id2verb[int(trainset[i]["verb"])] == "release":
        release_list.append(trainset[i])
    if id2verb[int(trainset[i]["verb"])] == "insert":
        insert_list.append(trainset[i])    

    if len(take_list) > 10 and len(open_list) > 10 and len(press_list) > 10 and len(release_list) > 50 and len(insert_list) > 10:
        break

In [ ]:
len(release_list)

In [ ]:
import cv2
import matplotlib.pyplot as plt

def extract_frames(video_path, indices):
    cap = cv2.VideoCapture(video_path)
    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        
    cap.release()
    return frames

def show_frames(frames, title):
    n = len(frames)
    cols = min(5, n)
    rows = (n + cols - 1) // cols

    plt.figure(figsize=(15, 3*rows))
    
    for i, frame in enumerate(frames):
        plt.subplot(rows, cols, i+1)
        plt.imshow(frame)
        plt.axis('off')
        plt.title(f"{title} #{i}")
    
    plt.tight_layout()
    plt.show()
    
def get_middle_frame(frames):
    return frames[len(frames)//2]

def show_side_by_side(frame1, frame2, title1, title2):
    plt.figure(figsize=(8,4))

    # Left
    plt.subplot(1, 2, 1)
    plt.imshow(frame1)
    plt.title(title1)
    plt.axis('off')

    # Right
    plt.subplot(1, 2, 2)
    plt.imshow(frame2)
    plt.title(title2)
    plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
verb1 = press_list[10]
# verb2 = insert_list[9]
verb2 = release_list[50]

verb1_frames = extract_frames(verb1["video_path"], verb1["indices"])
verb2_frames = extract_frames(verb2["video_path"], verb2["indices"])

# show_frames(verb1_frames, "press")
# show_frames(verb2_frames, "insert")

# Pick ONE frame (middle)
verb1_frame = get_middle_frame(verb1_frames)
verb2_frame = get_middle_frame(verb2_frames)

# Show side by side
show_side_by_side(verb1_frame, verb2_frame, "press", "release")

Explanations for parameters in DataLoader
- `pin_memory`: is used for fast memory accessing, but will consume memory on RAM

    Reference: https://docs.pytorch.org/tutorials/intermediate/pinmem_nonblock.html
- `drop_last`: is used to ignore the last batch if `len(dataset)` cannot be divisible by `batch_size`. If `drop_last=True` then the last batch will be keep even though it is smaller than `batch_size`

    Reference: https://discuss.pytorch.org/t/usage-of-drop-last-on-data-loader/66741


**Load dataloader**

In [ ]:
# from torch.utils.data import DataLoader
# from probing.dataloader import worker_init_fn, decode_clips

# trainloader = DataLoader(
#     trainset,
#     batch_size=20,
#     shuffle=True,
#     num_workers=3, # don't use num_workers bigger than 3 it will crash!
#     pin_memory=False, # set to True will read data faster especially when move data between cpu and gpu but will hit memory limit
#     drop_last=False,
#     worker_init_fn=worker_init_fn,
#     collate_fn=decode_clips,
# )
# # for data in dataloader:
# #     continue
# # print("no error occurs! great!")

Check buffer size

In [ ]:
# batch = next(iter(trainloader))

# buffer = batch["buffer"]
# size_gb = buffer.nelement() * buffer.element_size() / 1e9
# print("Buffer memory:", size_gb, "GB")

# Main

## Prepare common variables

### Load label dict

In [ ]:
# Load label dicts
from probing.finebio import FineBioDataset

id2label_dict = FineBioDataset.load_id2label_dict()

### Load config

In [ ]:
import yaml
from pprint import pprint
import torch

DEVICE = torch.device("cuda")

config_file = "configs/finebio_vivit_feature_extraction.yaml"
# === Load configs ===
with open(config_file, "r") as f:
    cfgs = yaml.safe_load(f)
pprint(cfgs)
dataloader_cfgs = cfgs["dataloader"]
data_cfgs = cfgs["data"]
opt_cfgs = cfgs["opt"]
classifier_cfgs = cfgs["classifier"]

## Load model

In [ ]:
# from probing.models import init_vivit
# model, processor = init_vivit()

In [ ]:
# del model, processor

Check model embedding size

In [ ]:
# print(model.embeddings.patch_embeddings.projection.out_channels)
# model.config.hidden_size

## Load classifier

In [ ]:
# load classifier
from probing.attentive_classifier import init_classifier
import torch


classifier = init_classifier(
    fuse=classifier_cfgs["fuse"],
    embed_dim=classifier_cfgs["embed_dim"],
    num_heads=classifier_cfgs["num_heads"],
    num_blocks=classifier_cfgs["num_heads"],
    num_verb_classes=id2label_dict["verb"]["num_classes"],
    num_manipulated_classes=id2label_dict["manipulated"]["num_classes"],
    num_affected_classes=id2label_dict["affected"]["num_classes"],
    device=DEVICE,
    checkpoint=None
)

## Load transform

In [ ]:
# from probing.transforms import VivitTransform
# print(processor.image_processor_type)
# print(DEVICE)
# transform = VivitTransform(processor=processor, device=DEVICE, use_bfloat16=True)

## Feature extraction

**Instead of training everytime, it best to save features from frozen backbone then train it with classifier.**

### Load trainset

In [ ]:
# # load dataset
# from probing.dataloader import init_data

# trainset, trainloader = init_data(split=["train"],
#                                 is_training=True,
#                                 fpc=dataloader_cfgs["collate_fn"]["fpc"],
#                                 annotation_file=data_cfgs["annotation_file"],
#                                 label_dict_dir="annotations",
#                                 video_dir=data_cfgs["video_dir"],
#                                 allow_clip_overlap=dataloader_cfgs["collate_fn"]["allow_clip_overlap"],
#                                 batch_size=dataloader_cfgs["train_batch_size"],
#                                 num_workers=dataloader_cfgs["train_num_workers"],
#                                 )

### Load valset

In [ ]:
# from probing.dataloader import init_data

# valset, valloader = init_data(split=["val"],
#                                 is_training=False,
#                                 fpc=dataloader_cfgs["collate_fn"]["fpc"],
#                                 annotation_file=data_cfgs["annotation_file"],
#                                 label_dict_dir="annotations",
#                                 video_dir=data_cfgs["video_dir"],
#                                 allow_clip_overlap=dataloader_cfgs["collate_fn"]["allow_clip_overlap"],
#                                 batch_size=dataloader_cfgs["val_batch_size"],
#                                 num_workers=dataloader_cfgs["val_num_workers"],
#                                 )

### Feature extraction loop

In [ ]:
# from probing.utils.feature_extraction.extract_loop import feature_extract_loop

# feature_extract_loop(
#     model=model, 
#     transform=transform, 
#     dataloader=trainloader, 
#     outdir=data_cfgs["train_feat_dir"], 
#     data_cfgs=data_cfgs, 
#     dataloader_cfgs=dataloader_cfgs
# )

# feature_extract_loop(
#     model=model, 
#     transform=transform, 
#     dataloader=valloader, 
#     outdir=data_cfgs["val_feat_dir"], 
#     data_cfgs=data_cfgs, 
#     dataloader_cfgs=dataloader_cfgs
# )

### Train with extracted feats

In [ ]:
# model.embeddings.patch_embeddings.num_patches

#### Main

In [ ]:
import torch.nn as nn
from probing.losses import sigmoid_focal_loss
from probing.optimizers import init_opt
from probing.utils.feature_extraction.dataloader import init_feat_data
from probing.utils.feature_extraction.eval import validate_classifier_one_epoch
from probing.utils.feature_extraction.train import train_classifier_one_epoch
import wandb
import time
import os

def training_classifier_loop(
    checkpoint_dir: str,
    id2label_dict: dict,
    num_epochs: int,
    classifier,
    data_cfgs,
    dataloader_cfgs,
    opt_cfgs,
    use_focal_loss: bool=False,
    save_freq: int=5,
):
    if not os.path.exists(checkpoint_dir):
        os.mkdir(checkpoint_dir)
    else:
        user_key = input(f"{checkpoint_dir} may contains features. Are you sure you want to overwrite it?[y/n]")
        if user_key.lower() != "y":
            raise RuntimeError("Change checkpoint_dir before starting!")

    # === load data ===
    # final batch size is multiplied by 16
    _, trainloader = init_feat_data(
        data_dir=data_cfgs["train_feat_dir"],
        batch_size=1,
        num_workers=4,
        shuffle=True,
        drop_last=False,
    )
    _, valloader = init_feat_data(
        data_dir=data_cfgs["val_feat_dir"],
        batch_size=1,
        num_workers=4,
        shuffle=False,
        drop_last=False,
    )

    # === define criterion ===
    if use_focal_loss:
        criterion = sigmoid_focal_loss
    else:
        criterion = nn.CrossEntropyLoss()
    
    # === load optimizer ===
    optimizer, scheduler, wd_scheduler, _ = init_opt(params=classifier, 
                         iterations_per_epoch=len(trainloader)*16,
                         num_epochs=10,
                         warmup=opt_cfgs["warmup"],
                         lr=opt_cfgs["lr"],
                         start_lr=opt_cfgs["lr"],
                         final_lr=opt_cfgs["lr"],
                         weight_decay=opt_cfgs["wd"],
                         final_weight_decay=opt_cfgs["wd"],
                         use_bfloat16=opt_cfgs["use_bfloat16"],#False
                         )
    
    # === init logger ===
    run = wandb.init(
        project="vivit-probing",
        name=f"debug-{int(time.time())}",
        mode="offline",
        reinit="finish_previous"
    )
    
    for epoch in range(num_epochs):
        # train
        train_classifier_one_epoch(
            dataloader=trainloader, 
            classifier=classifier, 
            criterion=criterion, 
            optimizer=optimizer,
            scheduler=scheduler,
            wd_scheduler=wd_scheduler,
            num_verb_classes=id2label_dict["verb"]["num_classes"],
            num_manipulated_classes=id2label_dict["manipulated"]["num_classes"],
            num_affected_classes=id2label_dict["affected"]["num_classes"],
            device=DEVICE,
            run=run
        )
        
        # === save model ===
        save_path = os.path.join(checkpoint_dir, f"checkpoint_{epoch+1}.pt")
        if (epoch + 1) % save_freq == 0:
            torch.save({
                "epoch": epoch,
                "classifier": classifier.state_dict(),
                "optimizer": optimizer.state_dict(),
            }, save_path)
        
        # val
        validate_classifier_one_epoch(
            dataloader=valloader, 
            classifier=classifier, 
            criterion=criterion, 
            num_verb_classes=id2label_dict["verb"]["num_classes"],
            num_manipulated_classes=id2label_dict["manipulated"]["num_classes"],
            num_affected_classes=id2label_dict["affected"]["num_classes"],
            device=DEVICE,
            run=run
        )
    
    # end loop
    run.finish()

In [ ]:
training_classifier_loop(
    id2label_dict=id2label_dict,
    num_epochs=cfgs["train_classifier"]["num_epochs"],
    classifier=classifier,
    data_cfgs=data_cfgs,
    dataloader_cfgs=dataloader_cfgs,
    opt_cfgs=opt_cfgs,
    use_focal_loss=True,
    checkpoint_dir="outputs/vivit/weights",
    save_freq=5,
)

In [ ]:
# # === save model ===
# checkpoint_dir = "outputs/vivit/weights"
# if not os.path.exists(checkpoint_dir):
#     os.mkdir(checkpoint_dir)
# save_path = os.path.join(checkpoint_dir, "checkpoint_1.pt")
# torch.save({
#     "epoch": 1,
#     "classifier": classifier.state_dict(),
# }, save_path)

# Online Training (no pre-extracted feats)

In [ ]:
from tqdm import tqdm
import torch
from probing.metrics import ClassMeanRecall

def train_classifier_one_epoch(
    device,
    model,
    transform,
    classifier,
    optimizer,
    scheduler,
    wd_scheduler,
    scaler,
    data_loader,
    num_verb_classes,
    num_manipulated_classes,
    num_affected_classes,
    criterion,
    run,
    use_bfloat16=True,
    log_every=10,            # ✅ NEW
):

    classifier.train()

    verb_metric_logger        = ClassMeanRecall(num_verb_classes, device=device, k=5)
    manipulated_metric_logger = ClassMeanRecall(num_manipulated_classes, device=device, k=5)
    affected_metric_logger    = ClassMeanRecall(num_affected_classes, device=device, k=5)

    total_loss = 0.0
    total_verb_loss = 0.0
    total_manipulated_loss = 0.0
    total_affected_loss = 0.0

    progress = tqdm(data_loader)

    for step, udata in enumerate(progress):
        
        clips = udata["buffer"].to(device)
        verb_labels = udata["verb"].long().to(device)
        manipulated_labels = udata["manipulated"].long().to(device)
        affected_labels = udata["affected"].long().to(device)

        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_bfloat16):
            with torch.no_grad():   # frozen backbone
                clips = transform(clips)
                features = model(clips, return_dict=True)
                # features: B, N, D
                # however N =  1 cls token + patch tokens
                # we don't need to include it in probing
                features = features.last_hidden_state[:,1:,:] # remove cls

            outputs = classifier(features)

            verb_loss = criterion(outputs["verb"], verb_labels)
            manipulated_loss = criterion(outputs["manipulated"], manipulated_labels)
            affected_loss = criterion(outputs["affected"], affected_labels)

            loss = verb_loss + manipulated_loss + affected_loss

        optimizer.zero_grad()

        if use_bfloat16:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        total_verb_loss += verb_loss.item()
        total_manipulated_loss += manipulated_loss.item()
        total_affected_loss += affected_loss.item()

        verb_metric_logger(outputs["verb"], verb_labels)
        manipulated_metric_logger(outputs["manipulated"], manipulated_labels)
        affected_metric_logger(outputs["affected"], affected_labels)

        # ✅ Print every N iterations
        if step % log_every == 0:
            progress.set_postfix({
                "loss": f"{loss.item():.4f}",
                "verb_loss": f"{verb_loss.item():.4f}",
                "manip_loss": f"{manipulated_loss.item():.4f}",
                "aff_loss": f"{affected_loss.item():.4f}",
            })
            run.log({
                "train/step_loss": loss.item(),
                "train/verb_loss_step": verb_loss.item(),
                "train/manipulated_loss_step": manipulated_loss.item(),
                "train/affected_loss_step": affected_loss.item(),
                "lr": optimizer.param_groups[0]["lr"],
            })

    scheduler.step()
    wd_scheduler.step()

    num_batches = step + 1  # important if max_iters used

    ret = dict(
        total_loss=total_loss / num_batches,
        verb=dict(
            loss=total_verb_loss / num_batches,
            **verb_metric_logger(outputs["verb"], verb_labels),
        ),
        manipulated=dict(
            loss=total_manipulated_loss / num_batches,
            **manipulated_metric_logger(outputs["manipulated"], manipulated_labels),
        ),
        affected=dict(
            loss=total_affected_loss / num_batches,
            **affected_metric_logger(outputs["affected"], affected_labels),
        ),
    )

    return ret

### Define validate

In [ ]:
@torch.no_grad()
def validate(
    device,
    model,
    classifier,
    data_loader,
    transform,
    num_verb_classes,
    num_manipulated_classes,
    num_affected_classes,
    criterion,
    run,
    # max_iters=None, # for quick debug
    log_every=10,
    use_bfloat16=True,
):

    classifier.eval()

    verb_metric_logger        = ClassMeanRecall(num_verb_classes, device=device, k=5)
    manipulated_metric_logger = ClassMeanRecall(num_manipulated_classes, device=device, k=5)
    affected_metric_logger    = ClassMeanRecall(num_affected_classes, device=device, k=5)


    total_loss = 0.0
    total_verb_loss = 0.0
    total_manipulated_loss = 0.0
    total_affected_loss = 0.0
    
    progress = tqdm(data_loader)
    for step, udata in enumerate(progress):
        # early break for debug
        # if max_iters is not None and step >= max_iters:
        #     break

        clips = udata["buffer"].to(device)
        clips = transform(clips)
        verb_labels = udata["verb"].long().to(device)
        manipulated_labels = udata["manipulated"].long().to(device)
        affected_labels = udata["affected"].long().to(device)

        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_bfloat16):
            features = model(clips, return_dict=True)
            # features: B, N, D
            # however N =  1 cls token + patch tokens
            # we don't need to include it in probing
            features = features.last_hidden_state[:,1:,:] # remove cls
            outputs = classifier(features)

            verb_loss = criterion(outputs["verb"], verb_labels)
            manipulated_loss = criterion(outputs["manipulated"], manipulated_labels)
            affected_loss = criterion(outputs["affected"], affected_labels)

            loss = verb_loss + manipulated_loss + affected_loss

        total_loss += loss.item()
        total_verb_loss += verb_loss.item()
        total_manipulated_loss += manipulated_loss.item()
        total_affected_loss += affected_loss.item()

        verb_metric_logger(outputs["verb"], verb_labels)
        manipulated_metric_logger(outputs["manipulated"], manipulated_labels)
        affected_metric_logger(outputs["affected"], affected_labels)

        
        # ✅ Print every N iterations
        if step % log_every == 0:
            progress.set_postfix({
                "loss": f"{loss.item():.4f}",
                "verb_loss": f"{verb_loss.item():.4f}",
                "manip_loss": f"{manipulated_loss.item():.4f}",
                "aff_loss": f"{affected_loss.item():.4f}",
            })
            run.log({
                "val/step_loss": loss.item(),
                "val/verb_loss_step": verb_loss.item(),
                "val/manipulated_loss_step": manipulated_loss.item(),
                "val/affected_loss_step": affected_loss.item(),
            })

    num_batches = step + 1 

    ret = dict(
        total_loss=total_loss / num_batches,
        verb=dict(
            loss=total_verb_loss / num_batches,
            **verb_metric_logger(outputs["verb"], verb_labels),
        ),
        manipulated=dict(
            loss=total_manipulated_loss / num_batches,
            **manipulated_metric_logger(outputs["manipulated"], manipulated_labels),
        ),
        affected=dict(
            loss=total_affected_loss / num_batches,
            **affected_metric_logger(outputs["affected"], affected_labels),
        ),
    )

    return ret

### Define training loop

In [ ]:
import torch.nn as nn
from probing.losses import sigmoid_focal_loss
import os
import wandb
import time

def train_model(
    device,
    model,
    classifier,
    train_loader,
    val_loader,
    transform,
    optimizer,
    scheduler,
    wd_scheduler,
    scaler,
    num_epochs,
    num_verb_classes,
    num_manipulated_classes,
    num_affected_classes,
    run,
    checkpoint_dir,
    use_focal_loss=True,
    # max_iters=None, # set to integer for quick debug
    log_every=10,
    save_freq = 5  # save model every $save_freq epochs
):
    if not os.path.exists(checkpoint_dir):
        os.mkdir(checkpoint_dir)
    else:
        user_key = input(f"{checkpoint_dir} may contains features. Are you sure you want to overwrite it?[y/n]")
        if user_key.lower() != "y":
            raise RuntimeError("Change checkpoint_dir before starting!")
    
    if use_focal_loss:
        criterion = sigmoid_focal_loss
    else:
        criterion = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):

        print(f"\n========== Epoch {epoch+1}/{num_epochs} ==========")
        print("\n--- Train ---")

        train_metrics = train_classifier_one_epoch(
            device=device,
            model=model,
            classifier=classifier,
            optimizer=optimizer,
            scheduler=scheduler,
            wd_scheduler=wd_scheduler,
            scaler=scaler,
            data_loader=train_loader,
            transform=transform, # for now use same transform for train/val/test
            num_verb_classes=num_verb_classes,
            num_manipulated_classes=num_manipulated_classes,
            num_affected_classes=num_affected_classes,
            criterion=criterion,
            # max_iters=max_iters,
            log_every=log_every,
            run=run,
        )

        print(train_metrics)
        print("\n--- Val ---")

        val_metrics = validate(
            device=device,
            model=model,
            classifier=classifier,
            data_loader=val_loader,
            transform=transform,
            num_verb_classes=num_verb_classes,
            num_manipulated_classes=num_manipulated_classes,
            num_affected_classes=num_affected_classes,
            criterion=criterion,
            # max_iters=max_iters,
            log_every=log_every,
            run=run,
        )

        print(val_metrics)
        
        run.log({
            "epoch": epoch,
            "train/total_loss": train_metrics["total_loss"],
            "val/total_loss": val_metrics["total_loss"],
        })
        
        # === save model ===
        save_path = os.path.join(checkpoint_dir, f"checkpoint_{epoch+1}.pt")
        if (epoch + 1) % save_freq == 0:
            torch.save({
                "epoch": epoch,
                "model": model.state_dict(),
                "classifier": classifier.state_dict(),
                "optimizer": optimizer.state_dict(),
            }, save_path)

### Train & Validate

In [ ]:
from probing.dataloader import init_data
from probing.optimizers import init_opt

TRAIN_BATCH_SIZE = 16 # using too big batch size can reduce performance
VAL_BATCH_SIZE = 16 # since val/test doesn't require grad so as long as your pc ram or gpu vram can load just use bigger batch size and num workers
NUM_EPOCHS = 40
LEARNING_RATE = 0.000425
WEIGHT_DECAY = 0.04
WARMUP_EPOCHS = 5
USE_BFLOAT16 = True
VIDEO_DIR = "data/FineBio/03 finebio_videos_fpv_all_w640"
config = {
    "train_batch_size": TRAIN_BATCH_SIZE,
    "val_batch_size": VAL_BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "num_workers": 2,

    "learning_rate": LEARNING_RATE,
    "start_lr": LEARNING_RATE,
    "final_lr": LEARNING_RATE,

    "weight_decay": WEIGHT_DECAY,
    "final_weight_decay": WEIGHT_DECAY,

    "warmup_epochs": WARMUP_EPOCHS,
    "use_bfloat16": USE_BFLOAT16,

    "dataset": "FineBio",
    "video_dir": VIDEO_DIR
}
train_dataset, train_loader = init_data(split=["train"], is_training=True,
                                        video_dir=VIDEO_DIR,
                                        num_workers=2,
                                        batch_size=TRAIN_BATCH_SIZE)
_, val_loader = init_data(
    split=["val"], is_training=False,
    video_dir=VIDEO_DIR,
    num_workers=2,
    batch_size=VAL_BATCH_SIZE)

optimizer, scheduler, wd_scheduler, scaler = init_opt(params=classifier,
                                                      iterations_per_epoch=len(train_dataset),
                                                      num_epochs=NUM_EPOCHS,
                                                      warmup=WARMUP_EPOCHS,
                                                      lr=LEARNING_RATE,
                                                      start_lr=LEARNING_RATE,
                                                      final_lr=LEARNING_RATE,
                                                      weight_decay=WEIGHT_DECAY,
                                                      final_weight_decay=WEIGHT_DECAY,
                                                      use_bfloat16=USE_BFLOAT16)

run = wandb.init(
        project="vivit-probing",
        name=f"debug-{int(time.time())}",
        mode="offline",
        reinit=True
    )

train_model(
    model=model,
    classifier=classifier,
    train_loader=train_loader,
    val_loader=val_loader,
    transform=transform,
    num_verb_classes=id2label_dict["verb"]["num_classes"],
    num_manipulated_classes=id2label_dict["manipulated"]["num_classes"],
    num_affected_classes=id2label_dict["affected"]["num_classes"],
    # opt
    optimizer=optimizer,
    scheduler=scheduler,
    wd_scheduler=wd_scheduler,
    scaler=scaler,
    use_focal_loss=True,
    num_epochs=NUM_EPOCHS,
    device=DEVICE,
    # max_iters=10,
    log_every=50,
)
run.finish()

In [ ]:
# torch.save({"epoch": 5,
#             "model": model.state_dict(),
#             "classifier": classifier.state_dict(),
#             "optimizer": optimizer.state_dict(),
#             }, f="outputs/vivit/checkpoint05.pt")

# Validate on testset

## Load classifier weight

In [ ]:
import torch
ckpt = torch.load("outputs/vivit/checkpoint05.pt", map_location="cpu")
classifier.load_state_dict(ckpt["classifier"])
classifier.cuda()
del ckpt

## Validate testset

In [ ]:
del all_labels, all_preds, batch, features, inputs, logits, test_loader

In [ ]:
from probing.dataloader import init_data
from tqdm import tqdm
# Put model and classifier to eval mode
model.eval()
classifier.eval()

# temporary parameters
USE_BFLOAT16 = True
TYPENAMES = ["verb", "manipulated", "affected"]

# load data
_, test_loader = init_data(split=["test"],
                           is_training=False,
                           batch_size=32,
                           num_workers=4)

all_preds = {}
all_labels = {}
for typename in TYPENAMES:
    all_preds[typename] = []
    all_labels[typename] = []

for batch in tqdm(test_loader):
    inputs = batch["buffer"].to(DEVICE)

    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BFLOAT16):
        with torch.no_grad():
            inputs = transform(inputs)
            
            features = model(inputs, return_dict=True) # shape: B, N, D where N= 1cls + patch tokens
            features = features.last_hidden_state[:,1:,:] # remove cls
            
            logits = classifier(features)
            
            for typename in TYPENAMES:
                all_preds[typename].append(torch.argmax(logits[typename], dim=1).cpu())
                all_labels[typename].append(batch[typename])

for typename in TYPENAMES:
    all_preds[typename] = torch.cat(all_preds[typename])
    all_labels[typename] = torch.cat(all_labels[typename])

## Visulize confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import json
import matplotlib.pyplot as plt
import seaborn as sns
from probing.finebio import FineBioDataset

TYPENAMES = ["verb", "manipulated", "affected"]
id2label_dict = {}
for typename in TYPENAMES:
    id2label_dict[typename] = FineBioDataset.load_label_dict(label_dict_dir="annotations", typename=typename, to_int=False)

def check_label_dict():
    for typename in id2label_dict:
        if "0" not in id2label_dict[typename]:
            print(id2label_dict)
            raise ValueError("Loaded dict is not int to label. We need int to label dict!")
check_label_dict()

def visualize_validation_results(typename):
    # get correct int to label dict, labels and preds for typename
    int_to_label_dict = id2label_dict[typename] 
    labels = all_labels[typename].numpy()
    preds = all_preds[typename].numpy()
    # convert back to words
    true_labels      = [int_to_label_dict[str(label)] for label in labels]
    predicted_labels = [int_to_label_dict[str(label)] for label in preds]

    # Get sorted unique class names
    class_names = sorted(list(set(true_labels)))

    cm = confusion_matrix(true_labels, predicted_labels, labels=class_names)

    plt.figure(figsize=(12, 10))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{typename} Confusion Matrix")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    from sklearn.metrics import classification_report

    print(classification_report(
        true_labels,
        predicted_labels,
        labels=class_names
    ))

for typename in TYPENAMES:
    visualize_validation_results(typename)